# Synthetic Data Generator

Generate synthetic datasets using multiple open-source LLMs with quantization for efficient GPU usage.

## Technologies
- **Models**: Llama-3.2-3B, Phi-4-mini, Gemma-3-270m, Qwen3-4B, DeepSeek-R1-Distill
- **Quantization**: 4-bit BitsAndBytes (nf4, bfloat16)
- **Framework**: HuggingFace Transformers
- **UI**: Gradio
- **Platform**: Google Colab (T4 GPU)

## Datasets
- Customer Profiles (13 fields)
- Product Catalogs (15 fields)
- Health Records (15 fields)

## Setup

In [ ]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6 gradio

In [ ]:
import torch
import json
import gc
import pandas as pd
import sys
from io import StringIO
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import gradio as gr

In [ ]:
# Check GPU
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
    print('Not connected to a GPU')
else:
    print(gpu_info)
    if gpu_info.find('Tesla T4') >= 0:
        print("Success - Connected to a T4")
    else:
        print(f"Connected to: {gpu_info.split('Tesla ')[1].split()[0] if 'Tesla' in gpu_info else 'Unknown GPU'}")

In [ ]:
# HuggingFace Login
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

## Models

In [ ]:
# Model configurations
MODELS = {
    "Llama-3.2-3B": "meta-llama/Llama-3.2-3B-Instruct",
    "Phi-4-mini": "microsoft/Phi-4-mini-instruct",
    "Gemma-3-270m": "google/gemma-3-270m-it",
    "Qwen3-4B": "Qwen/Qwen3-4B-Instruct-2507",
    "DeepSeek-R1": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
}

# Quantization config
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
def load_model(model_name):
    """Load model with quantization"""
    model_id = MODELS[model_name]
    print(f"Loading {model_name}...")
    
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.pad_token = tokenizer.eos_token
    
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        quantization_config=quant_config
    )
    
    memory = model.get_memory_footprint() / 1e6
    print(f"Memory footprint: {memory:,.1f} MB")
    
    return model, tokenizer

In [ ]:
def cleanup_model(model, tokenizer):
    """Clean up GPU memory"""
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print("GPU memory cleared")

## Dataset Generation

In [ ]:
DATASETS = {
    "Customer Profiles": {
        "fields": ["customer_id", "full_name", "email", "phone", "age", "gender", 
                  "address", "city", "country", "signup_date", "last_purchase_date", 
                  "total_spent", "preferred_category", "loyalty_status"],
        "prompt": "Generate exactly ONE realistic customer profile with the following fields: {fields}. \
            Return as a single JSON object, not an array. Do not generate multiple records."
    },
    "Product Catalogs": {
        "fields": ["product_id", "name", "category", "subcategory", "price", "description",
                  "sku", "stock_quantity", "brand", "manufacturer", "weight", 
                  "dimensions", "color", "material", "warranty_months"],
        "prompt": "Generate exactly ONE realistic product catalog entry with the following fields: {fields}. \
            Return as a single JSON object, not an array. Do not generate multiple records."
    },
    "Health Records": {
        "fields": ["patient_id", "name", "age", "gender", "blood_type", "weight_kg", 
                  "height_cm", "bmi", "blood_pressure_systolic", "blood_pressure_diastolic",
                  "heart_rate", "temperature_c", "allergies", "medications", "conditions"],
        "prompt": "Generate exactly ONE realistic health record with the following fields: {fields}. \
            Return as a single JSON object, not an array. Do not generate multiple records."
    }
}

In [ ]:
def generate_record(model, tokenizer, dataset_type, custom_prompt=None):
    """Generate a single record with streaming"""
    dataset_info = DATASETS[dataset_type]
    fields = dataset_info["fields"]
    base_prompt = dataset_info["prompt"].format(fields=", ".join(fields))
    
    prompt = custom_prompt if custom_prompt else base_prompt
    
    messages = [
        {"role": "system", "content": "You are a data generator. Return valid JSON only. \
            Generate exactly ONE record as a single JSON object, not an array."},
        {"role": "user", "content": prompt}
    ]
    
    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
    
    # Use streaming for real-time output
    streamer = TextStreamer(tokenizer)
    outputs = model.generate(inputs, max_new_tokens=500, temperature=0.7, streamer=streamer)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract JSON from response
    try:
        json_start = response.find('{')
        json_end = response.rfind('}') + 1
        if json_start != -1 and json_end != 0:
            json_str = response[json_start:json_end]
            data = json.loads(json_str)
            
            # If it's an array, take the first item
            if isinstance(data, list):
                return data[0] if data else None
            return data
    except (json.JSONDecodeError, ValueError):
        pass

In [ ]:
def generate_batch(model, tokenizer, dataset_type, quantity, custom_prompt=None, progress=None):
    """Generate multiple records with history to avoid duplicates"""
    records = []
    
    for i in range(quantity):
        if i % 10 == 0:
            if progress:
                progress((i + 1) / quantity, desc=f"Generating record {i+1}/{quantity}")
        
        # Build prompt with history
        if custom_prompt:
            prompt = custom_prompt
        else:
            dataset_info = DATASETS[dataset_type]
            fields = dataset_info["fields"]
            base_prompt = dataset_info["prompt"].format(fields=", ".join(fields))
            
            # Add history to avoid duplicates
            if records:
                history_text = "\n\nPreviously generated records (do not repeat):\n"
                for record in records[-3:]:
                    history_text += f"- {record.get('full_name', record.get('name', 'unknown'))}\n"
                prompt = f"{base_prompt}\n{history_text}"
            else:
                prompt = base_prompt
        
        record = generate_record(model, tokenizer, dataset_type, prompt)
        if record:
            records.append(record)
    
    return pd.DataFrame(records)

## Gradio UI

In [ ]:
def generate_data(dataset_type, model_name, quantity, custom_prompt, progress=gr.Progress()):
    """Main generation function for Gradio"""
    import tempfile
    
    try:
        # Capture HuggingFace progress
        old_stdout = sys.stdout
        sys.stdout = captured_output = StringIO()
        
        # Load model
        progress(0.1, desc="Loading model from HuggingFace...")
        model, tokenizer = load_model(model_name)
        
        # Restore stdout and get captured output
        sys.stdout = old_stdout
        hf_progress = captured_output.getvalue()
        if hf_progress:
            print(hf_progress)
        
        progress(0.3, desc="Model loaded successfully")
        
        # Generate batch
        progress(0.5, desc=f"Generating {quantity} records...")
        df = generate_batch(model, tokenizer, dataset_type, quantity, custom_prompt, progress)
        progress(0.9, desc=f"Generated {len(df)} records")
        
        # Cleanup
        cleanup_model(model, tokenizer)
        progress(1.0, desc="GPU memory cleared")
        
        if df.empty:
            return "No valid records generated", None, None
        
        # Save to temp file for download
        temp_file = tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False)
        df.to_csv(temp_file.name, index=False)
        temp_file.close()
        
        return f"Generated {len(df)} records", df, temp_file.name
        
    except Exception as e:
        sys.stdout = old_stdout
        return f"Error: {str(e)}", None, None

In [ ]:
# Create Gradio interface
with gr.Blocks(title="Synthetic Data Generator") as demo:
    gr.Markdown("# Synthetic Data Generator")
    gr.Markdown("Generate synthetic datasets using multiple open-source LLMs with 4-bit quantization.")
    
    with gr.Row():
        with gr.Column():
            dataset_type = gr.Dropdown(
                choices=list(DATASETS.keys()),
                value="Customer Profiles",
                label="Dataset Type"
            )
            model_name = gr.Dropdown(
                choices=list(MODELS.keys()),
                value="Llama-3.2-3B",
                label="Model"
            )
            quantity = gr.Slider(
                minimum=1,
                maximum=10,
                value=3,
                step=1,
                label="Quantity"
            )
            custom_prompt = gr.Textbox(
                label="Custom Prompt (optional)",
                placeholder="Override default prompt..."
            )
            generate_btn = gr.Button("Generate Data", variant="primary")
        
        with gr.Column():
            output_status = gr.Textbox(label="Status")
            output_dataframe = gr.Dataframe(label="Generated Data Preview")
            download_btn = gr.DownloadButton(label="Download CSV", visible=False)
    
    generate_btn.click(
        fn=generate_data,
        inputs=[dataset_type, model_name, quantity, custom_prompt],
        outputs=[output_status, output_dataframe, download_btn]
    )
    
    gr.Examples(
        examples=[
            ["Customer Profiles", "Llama-3.2-3B", 3, ""],
            ["Product Catalogs", "Phi-4-mini", 5, ""],
            ["Health Records", "Qwen3-4B", 2, ""]
        ],
        inputs=[dataset_type, model_name, quantity, custom_prompt]
    )

## Launch

In [ ]:
# Launch Gradio interface
if __name__ == "__main__":
    demo.launch(share=True, debug=True)